# Dual-Engine Rule Validation: LLM vs Z3

This notebook runs the **same rules** through both engines for direct comparison:
- **`invoice_math_llm`** → LLM engine (semantic, non-deterministic)
- **`invoice_math_z3`** → Z3 engine (formal, deterministic)

Same rules, same data, different engines. Compare the results.

## 1. Setup

In [ ]:
%load_ext autoreload
%autoreload 2

#ROOTDIR = "../"
#%pip uninstall -y idp_common 2>/dev/null
#%pip install -e "$ROOTDIR[rule_validation]" -q
#%pip install z3-solver pyyaml python-dotenv -q

In [ ]:
import os
import json
import time
import yaml
import boto3
import logging
import datetime

os.environ["AWS_REGION"] = "us-west-2"
os.environ["AWS_PROFILE"] = "default"

from idp_common.models import Document, Status, Section, Page
from idp_common import s3
from dotenv import load_dotenv
load_dotenv()

# Logging — show rule_validation at DEBUG to see engine routing
logging.basicConfig(level=logging.WARNING)
logging.getLogger('idp_common.rule_validation').setLevel(logging.DEBUG)

os.environ['METRIC_NAMESPACE'] = 'IDP-Notebook-DualEngine'

sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']
region = os.environ['AWS_REGION']

input_bucket = os.getenv('IDP_INPUT_BUCKET_NAME', f'idp-notebook-input-{account_id}-{region}')
output_bucket = os.getenv('IDP_OUTPUT_BUCKET_NAME', f'idp-notebook-output-{account_id}-{region}')

print(f'Region: {region}')
print(f'Input bucket: {input_bucket}')
print(f'Output bucket: {output_bucket}')


## 2. Create a synthetic invoice document

Instead of running OCR on a real PDF, we create a `Document` with synthetic
page text and extraction results. This lets us test the rule validation
engines without needing Textract.

In [ ]:
s3_client = boto3.client('s3')

def ensure_bucket(name):
    try:
        s3_client.head_bucket(Bucket=name)
    except Exception:
        params = {'Bucket': name}
        if region != 'us-east-1':
            params['CreateBucketConfiguration'] = {'LocationConstraint': region}
        s3_client.create_bucket(**params)
        s3_client.get_waiter('bucket_exists').wait(Bucket=name)
    print(f'Bucket ready: {name}')

ensure_bucket(input_bucket)
ensure_bucket(output_bucket)

In [ ]:
# Synthetic invoice text (what OCR + parsing would produce)
invoice_text = """INVOICE

Vendor: Acme Supplies Inc.
Invoice Number: INV-2026-0042
Date: April 1, 2026

Description          Qty    Unit Price    Amount
Widget A              10       $25.00     $250.00
Widget B               5       $40.00     $200.00
Service Fee            1      $100.00     $100.00

Subtotal: $550.00
Tax (13% HST): $71.50
Total: $621.50
"""

# Structured extraction results (what the extraction service would produce)
extraction_result = {
    "inference_result": {
        "vendor_name": "Acme Supplies Inc.",
        "invoice_number": "INV-2026-0042",
        "invoice_date": "April 1, 2026",
        "subtotal": "550.00",
        "tax_amount": "71.50",
        "total_amount": "621.50"
    }
}

# Upload to S3 (mimicking what OCR + extraction services would do)
doc_prefix = "dual-engine-test/invoice-test"

parsed_text_key = f"{doc_prefix}/pages/1/result.md"
s3.write_content(invoice_text, output_bucket, parsed_text_key, content_type='text/plain')

extraction_key = f"{doc_prefix}/sections/1/result.json"
s3.write_content(extraction_result, output_bucket, extraction_key, content_type='application/json')

print(f'Uploaded parsed text to: s3://{output_bucket}/{parsed_text_key}')
print(f'Uploaded extraction to:  s3://{output_bucket}/{extraction_key}')

In [ ]:
# Build the Document object as if OCR + classification + extraction already ran
document = Document(
    id="invoice-dual-engine-test",
    input_bucket=input_bucket,
    input_key=doc_prefix,
    output_bucket=output_bucket,
    num_pages=1,
    status=Status.RUNNING,
    pages={
        "1": Page(
            page_id="1",
            parsed_text_uri=f"s3://{output_bucket}/{parsed_text_key}",
            classification="invoice",
            confidence=0.99,
        )
    },
    sections=[
        Section(
            section_id="1",
            classification="invoice",
            confidence=0.99,
            page_ids=["1"],
            extraction_result_uri=f"s3://{output_bucket}/{extraction_key}",
        )
    ],
)

print(f'Document ID: {document.id}')
print(f'Sections: {len(document.sections)}')
print(f'Pages: {list(document.pages.keys())}')
print(f'Extraction URI: {document.sections[0].extraction_result_uri}')

In [ ]:
document

## 3. Load dual-engine config

This config has the **same two rules** defined twice:
- `invoice_math_llm` → LLM engine (default)
- `invoice_math_z3` → Z3 engine (`x-aws-idp-validation-engine: z3`)

Both check: tax_calculation and tax_rate_check.

In [ ]:
config_path = "rule-validation-config/config_dual_engine.yaml"
with open(config_path, 'r') as f:
    config_data = yaml.safe_load(f)

print("Policy classes:")
for pc in config_data.get('policy_classes', []):
    policy_type = pc.get('x-aws-idp-policy-type')
    rule_props = pc.get('rule_properties', {})
    rules_info = []
    for name, prop in rule_props.items():
        engine = prop.get('x-aws-idp-validation-engine', 'llm')
        rules_info.append(f'{name}({engine})')
    print(f"  {policy_type}: {rules_info}")

## 4. Run Rule Validation (both engines)

The service routes each rule_type to the correct engine. Watch the logs for:
- `Processed rule type invoice_math_llm via llm engine`
- `Processed rule type invoice_math_z3 via z3 engine`

In [ ]:
from idp_common.rule_validation import RuleValidationService
from idp_common.models import RuleValidationResult

# Set matched_policy_types so the service knows which policy types to validate
policy_types = [pc.get('x-aws-idp-policy-type') for pc in config_data.get('policy_classes', [])]
document.rule_validation_result = RuleValidationResult(request_id=document.id, matched_policy_types=policy_types)

rule_validation_service = RuleValidationService(
    region=region,
    config=config_data,
)

print("Running rule validation (LLM + Z3)...")
print(f"Matched policy types: {policy_types}")
start_time = time.time()

validated_document = rule_validation_service.validate_document(document)

duration = time.time() - start_time
print(f"\nCompleted in {duration:.2f}s")
print(f"Status: {validated_document.status.value}")
print(f"Errors: {validated_document.errors}")

In [ ]:
# Load section results from S3
result_uri = validated_document.rule_validation_result.output_uri
print(f"Results URI: {result_uri}")

section_data = s3.get_json_content(result_uri)

for rule_type, responses in section_data.get("responses", {}).items():
    # Determine engine from config
    engine = "llm"
    for pc in config_data.get("policy_classes", []):
        if pc.get("x-aws-idp-policy-type") == rule_type:
            # Check per-rule engine (new format: engine is on each rule property)
            rule_props = pc.get("rule_properties", {})
            engines = set(p.get("x-aws-idp-validation-engine", "llm") for p in rule_props.values())
            engine = engines.pop() if len(engines) == 1 else "mixed"
            break

    print(f"{"="*60}")
    print(f"Rule Type: {rule_type}  [engine: {engine}]")
    print(f"{"="*60}")

    if isinstance(responses, list):
        for resp in responses:
            print(f"  Rule: {resp.get("rule", "N/A")[:80]}")
            if engine == "z3":
                # Z3 returns recommendation + reasoning directly
                print(f"  Recommendation: {resp.get("recommendation")}")
                print(f"  Reasoning: {resp.get("reasoning", "")[:200]}")
            else:
                # LLM fact extraction returns extracted_facts (step 1 of 2)
                # Recommendation comes after orchestrator runs (step 2)
                facts = resp.get("extracted_facts", [])
                print(f"  Extracted Facts: {len(facts)} found")
                for fact in facts[:3]:
                    print(f"    - {fact.get("fact", "")[:100]}")
                    print(f"      Citation: {fact.get("citation", "")}")
                summary = resp.get("extraction_summary", "")
                if summary:
                    print(f"  Summary: {summary[:200]}")
            print()


In [ ]:
# Unified results view (raw section results before orchestrator)
# Note: LLM rules only have extracted_facts at this stage (recommendation comes from orchestrator)
# Z3 rules already have recommendation + reasoning

result_uri = validated_document.rule_validation_result.output_uri
# The orchestrator may have changed output_uri to .md; use section results directly
section_uri = f's3://{output_bucket}/{doc_prefix}/rule_validation/sections/section_1_responses.json'
print(f"Section Results URI: {section_uri}\n")

section_data = s3.get_json_content(section_uri)

for rule_type, responses in section_data.get('responses', {}).items():
    # Determine engine from config
    engine = 'llm'
    for pc in config_data.get('policy_classes', []):
        if pc.get('x-aws-idp-policy-type') == rule_type:
            rule_props = pc.get('rule_properties', {})
            engines = set(p.get('x-aws-idp-validation-engine', 'llm') for p in rule_props.values())
            engine = engines.pop() if len(engines) == 1 else 'mixed'
            break

    print(f"{'='*60}")
    print(f"Rule Type: {rule_type}  [engine: {engine}]")
    print(f"{'='*60}")

    if isinstance(responses, list):
        for resp in responses:
            print(f"  Rule: {resp.get('rule', 'N/A')[:80]}")
            recommendation = resp.get('recommendation')
            if recommendation:
                print(f"  Recommendation: {recommendation}")
                print(f"  Reasoning: {resp.get('reasoning', '')[:200]}")
            else:
                # LLM step 1: only facts extracted (recommendation comes from orchestrator step 2)
                facts = resp.get('extracted_facts', [])
                print(f"  [Step 1 only] Extracted Facts: {len(facts)} found")
                print(f"  (Run orchestrator or comparison cell for final LLM recommendation)")
            print()


## 6. Run Orchestrator (consolidation)

The orchestrator consolidates results from both engines into a single
summary. For Z3 rules, the recommendation is already available from step 1.
For LLM rules, the orchestrator runs a second LLM call to convert
extracted facts into a final recommendation.

In [ ]:
from idp_common.rule_validation import RuleValidationOrchestratorService

orchestrator = RuleValidationOrchestratorService(config=config_data)

# multiple_sections=True forces the orchestrator LLM call to run,
# which is required for the fact-extraction workflow (step 2: facts → recommendation)
document = orchestrator.consolidate_and_save(
    document=validated_document,
    config=config_data,
    multiple_sections=True,
)

print(f"Consolidated URI: {document.rule_validation_result.output_uri}")
print(f"Sections processed: {document.rule_validation_result.metadata.get("sections_processed", 0)}")


In [ ]:
# Load and display the consolidated summary
# output_uri may point to .md — swap to .json
summary_uri = document.rule_validation_result.output_uri
if summary_uri.endswith(".md"):
    summary_uri = summary_uri.replace(".md", ".json")

print(f"Loading: {summary_uri}")
summary = s3.get_json_content(summary_uri)

print("Overall Statistics:")
print(json.dumps(summary.get("overall_statistics", {}), indent=2))

print("Per Rule Type:")
for rule_type, details in summary.get("rule_details", {}).items():
    print(f"  {rule_type}: {details.get("recommendation_counts", {})}")


In [ ]:
# === Side-by-Side Comparison: Z3 vs LLM (full recommendation) ===
# Z3 produces recommendation directly. For LLM, we call the orchestrator's
# reasoning LLM on the extracted facts to get the final recommendation.

from idp_common import bedrock

# Load section results
# Load from section results (before orchestrator consolidation)
section_results_uri = f's3://{output_bucket}/{doc_prefix}/rule_validation/sections/section_1_responses.json'
section_data = s3.get_json_content(section_results_uri)

# Orchestrator reasoning prompt (same as the service uses)
reasoning_system_prompt = config_data['rule_validation']['rule_validation_orchestrator']['system_prompt']
reasoning_task_prompt = config_data['rule_validation']['rule_validation_orchestrator']['task_prompt']
recommendation_options = config_data['rule_validation'].get('recommendation_options', '')
model_id = config_data['rule_validation']['rule_validation_orchestrator']['model']

comparison_results = []

for rule_type, responses in section_data.get('responses', {}).items():
    if not isinstance(responses, list):
        continue
    for resp in responses:
        rule = resp.get('rule', 'N/A')
        # Determine engine
        engine = 'llm'
        for pc in config_data.get('policy_classes', []):
            if pc.get('x-aws-idp-policy-type') == rule_type:
                rule_props = pc.get('rule_properties', {})
                for prop in rule_props.values():
                    if prop.get('description', '').startswith(rule[:30]):
                        engine = prop.get('x-aws-idp-validation-engine', 'llm')
                        break
                break

        if engine == 'z3':
            # Z3 already has recommendation
            comparison_results.append({
                'rule': rule[:80],
                'engine': 'Z3',
                'recommendation': resp.get('recommendation'),
                'reasoning': resp.get('reasoning', '')[:150],
            })
        else:
            # LLM: call orchestrator reasoning on extracted facts
            facts = resp.get('extracted_facts', [])
            evidence_text = json.dumps(facts, indent=2)
            prompt = reasoning_task_prompt.replace('{extracted_evidence}', evidence_text)
            prompt = prompt.replace('{recommendation_options}', recommendation_options)
            prompt = prompt.replace('{rule_type}', rule_type)
            prompt = prompt.replace('{rule}', rule)
            prompt = prompt.replace('<<CACHEPOINT>>', '')

            llm_response = bedrock.invoke_model(
                model_id=model_id,
                system_prompt=reasoning_system_prompt,
                content=[{'text': prompt}],
                temperature=0.0,
                context='RuleValidation-Comparison',
            )
            response_text = bedrock.extract_text_from_response(llm_response)
            # Parse response
            try:
                if '<response>' in response_text:
                    response_text = response_text.split('<response>')[1].split('</response>')[0].strip()
                result_dict = json.loads(response_text)
                comparison_results.append({
                    'rule': rule[:80],
                    'engine': 'LLM',
                    'recommendation': result_dict.get('recommendation'),
                    'reasoning': result_dict.get('reasoning', '')[:150],
                })
            except Exception as e:
                comparison_results.append({
                    'rule': rule[:80],
                    'engine': 'LLM',
                    'recommendation': 'Parse Error',
                    'reasoning': str(e)[:150],
                })

# Display comparison table
print(f"{'='*90}")
print(f"{'SIDE-BY-SIDE COMPARISON: Z3 vs LLM':^90}")
print(f"{'='*90}")
print(f"{'Rule':<50} {'Engine':<6} {'Recommendation':<20}")
print(f"{'-'*90}")
for r in comparison_results:
    print(f"{r['rule']:<50} {r['engine']:<6} {r['recommendation']:<20}")
print(f"{'-'*90}")
print()
print('Detailed Reasoning:')
for r in comparison_results:
    print(f"  [{r['engine']}] {r['rule'][:60]}")
    print(f"       → {r['reasoning']}")
    print()


## 7. Direct Z3 engine test (standalone)

You can also use the Z3 engine directly, outside the IDP pipeline,
for quick testing with structured or unstructured data.
This gives a proper side-by-side comparison since Z3 produces
recommendation + reasoning in a single step (no orchestrator needed).

In [ ]:
from idp_common.rule_validation.z3_engine import Z3EngineAdapter

z3_adapter = Z3EngineAdapter()

# --- Test with STRUCTURED data (extraction results) ---
print("=" * 50)
print("Test 1: Z3 with structured extraction results")
print("=" * 50)

structured_data = {
    "inference_result": {
        "subtotal": "550.00",
        "tax_amount": "71.50",
        "total_amount": "621.50",
    }
}

result = z3_adapter.validate_rule(
    rule_description="The total amount must equal the subtotal plus the tax amount.",
    rule_type="invoice_math",
    extraction_results=structured_data,
    document_text="",  # not needed when structured data is available
)
print(f"Recommendation: {result['recommendation']}")
print(f"Reasoning: {result['reasoning']}")

In [ ]:
# --- Test with UNSTRUCTURED data (raw text only) ---
print("=" * 50)
print("Test 2: Z3 with unstructured text")
print("=" * 50)

raw_text = """Invoice from Acme Corp.
Subtotal: $550.00
Tax (13%): $71.50
Total Due: $621.50"""

result = z3_adapter.validate_rule(
    rule_description="The total amount must equal the subtotal plus the tax amount.",
    rule_type="invoice_math",
    extraction_results={},  # no structured data
    document_text=raw_text,
)
print(f"Recommendation: {result['recommendation']}")
print(f"Reasoning: {result['reasoning']}")

In [ ]:
# --- Test with FAILING data (math doesn't add up) ---
print("=" * 50)
print("Test 3: Z3 with data that should FAIL")
print("=" * 50)

bad_text = """Invoice from Bad Corp.
Subtotal: $550.00
Tax: $71.50
Total Due: $700.00"""

result = z3_adapter.validate_rule(
    rule_description="The total amount must equal the subtotal plus the tax amount.",
    rule_type="invoice_math",
    extraction_results={},
    document_text=bad_text,
)
print(f"Recommendation: {result['recommendation']}")
print(f"Reasoning: {result['reasoning']}")

## Summary

| Engine | Rule Type | Rules | How |
|--------|-----------|-------|-----|
| LLM | `invoice_math_llm` | tax_calculation, tax_rate_check | Semantic analysis via Bedrock |
| Z3 | `invoice_math_z3` | tax_calculation, tax_rate_check | Formal constraint solving |

Same rules, same data — compare recommendations and reasoning between engines.

---
## 8. Complex Example: Condo Fee Validation (Mortgage Underwriting)

This is a real-world mortgage underwriting rule with:
- Conditional logic (condo vs non-condo)
- Percentage thresholds with exceptions (GDS >= 35%, credit director override)
- Minimum fee floors ($200)
- Asymmetric tolerance (SAP can be higher, but not >5% lower)

We test the **same rule** on both engines with a synthetic condo document.

In [ ]:
# Synthetic condo document text (what OCR would produce)
condo_text = """CONDOMINIUM ASSOCIATION STATEMENT

Property: 4275 Rue Saint-Denis, Apt 812, Montreal, QC H2J 2K8
Property Type: Residential Condominium
Unit Owner: Emilie Tanguay and Marc-Andre Tanguay

Monthly Condo Fees: $450.00
Includes: Common area maintenance, building insurance, reserve fund contribution
Does NOT include: Heating, parking

Contingency Fund Balance: $125,000
Number of Units: 48

SAP SYSTEM RECORD
SAP Condo Fee: $435.00
GDS Ratio: 32.5%
Credit Director Exception: None
"""

# Structured extraction results (what extraction service would produce)
condo_extraction = {
    "inference_result": {
        "property_type": "Residential Condominium",
        "documented_condo_fee": "450.00",
        "sap_condo_fee": "435.00",
        "gds_ratio": "32.5",
        "credit_director_exception": "None",
        "property_address": "4275 Rue Saint-Denis, Apt 812, Montreal, QC H2J 2K8",
    }
}

# Upload to S3
condo_prefix = "dual-engine-test/condo-test"

condo_text_key = f"{condo_prefix}/pages/1/result.md"
s3.write_content(condo_text, output_bucket, condo_text_key, content_type='text/plain')

condo_extraction_key = f"{condo_prefix}/sections/1/result.json"
s3.write_content(condo_extraction, output_bucket, condo_extraction_key, content_type='application/json')

print('Condo document uploaded to S3')

In [ ]:
condo_document = Document(
    id="condo-fee-validation-test",
    input_bucket=input_bucket,
    input_key=condo_prefix,
    output_bucket=output_bucket,
    num_pages=1,
    status=Status.RUNNING,
    pages={
        "1": Page(
            page_id="1",
            parsed_text_uri=f"s3://{output_bucket}/{condo_text_key}",
            classification="condo_association_statement",
            confidence=0.99,
        )
    },
    sections=[
        Section(
            section_id="1",
            classification="condo_association_statement",
            confidence=0.99,
            page_ids=["1"],
            extraction_result_uri=f"s3://{output_bucket}/{condo_extraction_key}",
        )
    ],
)

print(f'Document: {condo_document.id}')

In [ ]:
# Add condo fee rules to the existing config (both engines)
condo_rule_description = (
    "This rule is for condos only. "
    "Condo fee must match SAP within 5% variance. "
    "The condo fee cannot be less than $200. "
    "SAP condo fees must be greater than or equal to 95% of the documented condo fees. "
    "SAP can be any amount higher than documented value, but cannot be more than 5% lower. "
    "Do not accept the 5% variance when: "
    "1) GDS is equal to or more than 35% or "
    "2) an exception was approved by credit director. "
    "In those cases SAP must match documented fees exactly. "
    "Condo fees must be at least $200. "
    "If property is not a condominium, outcome is pass."
)

condo_config = yaml.safe_load(open('rule-validation-config/config_dual_engine.yaml'))

# Replace policy_classes with condo rules for both engines
condo_config['policy_classes'] = [
    {
        '$schema': 'https://json-schema.org/draft/2020-12/schema',
        'x-aws-idp-policy-type': 'condo_fee_llm',
        'type': 'object',
        'rule_properties': {
            'condo_fee_validation': {
                'type': 'string',
                'description': condo_rule_description,
                'x-aws-idp-validation-engine': 'llm',
            }
        },
        '$id': 'condo_fee_llm',
    },
    {
        '$schema': 'https://json-schema.org/draft/2020-12/schema',
        'x-aws-idp-policy-type': 'condo_fee_z3',
        'type': 'object',
        'rule_properties': {
            'condo_fee_validation': {
                'type': 'string',
                'description': condo_rule_description,
                'x-aws-idp-validation-engine': 'z3',
            }
        },
        '$id': 'condo_fee_z3',
    },
]

print('Policy classes:')
for pc in condo_config['policy_classes']:
    policy_type = pc['x-aws-idp-policy-type']
    rule_props = pc.get('rule_properties', {})
    engines = [p.get('x-aws-idp-validation-engine', 'llm') for p in rule_props.values()]
    print(f"  {policy_type} [{engines[0]}]")
print(f'\nRule: {condo_rule_description[:100]}...')

In [ ]:
# Run rule validation with both engines
# Set matched_policy_types for condo document
condo_policy_types = [pc.get('x-aws-idp-policy-type') for pc in condo_config.get('policy_classes', [])]
condo_document.rule_validation_result = RuleValidationResult(request_id=condo_document.id, matched_policy_types=condo_policy_types)

condo_service = RuleValidationService(region=region, config=condo_config)

print('Running condo fee validation (LLM + Z3)...')
start_time = time.time()
condo_result = condo_service.validate_document(condo_document)
print(f'Completed in {time.time() - start_time:.2f}s')
print(f'Errors: {condo_result.errors}')

In [ ]:
# Inspect section results
result_uri = condo_result.rule_validation_result.output_uri
section_data = s3.get_json_content(result_uri)

for rule_type, responses in section_data.get('responses', {}).items():
    engine = 'z3' if 'z3' in rule_type else 'llm'
    print(f"{'='*60}")
    print(f"Rule Type: {rule_type}  [engine: {engine}]")
    print(f"{'='*60}")
    if isinstance(responses, list):
        for resp in responses:
            print(f"  Rule: {resp.get('rule', 'N/A')[:80]}...")
            if engine == 'z3':
                print(f"  Recommendation: {resp.get('recommendation')}")
                print(f"  Reasoning: {resp.get('reasoning', '')[:300]}")
            else:
                facts = resp.get('extracted_facts', [])
                print(f"  Extracted Facts: {len(facts)}")
                for fact in facts:
                    print(f"    - {fact.get('fact', '')[:120]}")
            print()

In [ ]:
# Run orchestrator for final LLM recommendations
condo_orchestrator = RuleValidationOrchestratorService(config=condo_config)
condo_final = condo_orchestrator.consolidate_and_save(
    document=condo_result,
    config=condo_config,
    multiple_sections=True,
)

# Load consolidated summary
summary_uri = condo_final.rule_validation_result.output_uri
if summary_uri.endswith('.md'):
    summary_uri = summary_uri.replace('.md', '.json')

summary = s3.get_json_content(summary_uri)

print('Overall Statistics:')
print(json.dumps(summary.get('overall_statistics', {}), indent=2))
print()
print('Per Rule Type:')
for rule_type, details in summary.get('rule_details', {}).items():
    print(f"  {rule_type}: {details.get('recommendation_counts', {})}")
    for rule_item in details.get('rules', []):
        print(f"    Rule: {rule_item.get('rule', '')[:80]}...")
        print(f"    Recommendation: {rule_item.get('recommendation')}")
        print(f"    Reasoning: {rule_item.get('reasoning', '')[:200]}")
        print()

### Analysis

In this scenario:
- Documented condo fee: **$450**
- SAP condo fee: **$435** (96.7% of documented — within 5%)
- GDS ratio: **32.5%** (below 35% threshold)
- No credit director exception

Expected result: **Pass** — SAP is $435 which is ≥ 95% of $450 ($427.50),
the 5% variance is allowed (GDS < 35%, no exception), and fees are above $200.

The Z3 engine should reach this conclusion deterministically via constraint solving.
The LLM engine should reach the same conclusion via semantic reasoning over the extracted facts.

In [ ]:
# Direct Z3 test with data that should FAIL
# SAP fee is $400 = 88.9% of $450 (below 95% threshold)
print('=' * 60)
print('Test: Condo fee FAIL case (SAP too low)')
print('=' * 60)

fail_result = z3_adapter.validate_rule(
    rule_description=condo_rule_description,
    rule_type='condo_fee_z3',
    extraction_results={},
    document_text=(
        'Property Type: Residential Condominium\n'
        'Documented Condo Fee: $450.00\n'
        'SAP Condo Fee: $400.00\n'
        'GDS Ratio: 32%\n'
        'Credit Director Exception: None'
    ),
)
print(f"Recommendation: {fail_result['recommendation']}")
print(f"Reasoning: {fail_result['reasoning']}")

print()
print('=' * 60)
print('Test: Condo fee FAIL case (GDS >= 35%, exact match required)')
print('=' * 60)

# SAP is $435 (within 5%) but GDS is 36% so exact match required
fail_result2 = z3_adapter.validate_rule(
    rule_description=condo_rule_description,
    rule_type='condo_fee_z3',
    extraction_results={},
    document_text=(
        'Property Type: Residential Condominium\n'
        'Documented Condo Fee: $450.00\n'
        'SAP Condo Fee: $435.00\n'
        'GDS Ratio: 36%\n'
        'Credit Director Exception: None'
    ),
)
print(f"Recommendation: {fail_result2['recommendation']}")
print(f"Reasoning: {fail_result2['reasoning']}")

print()
print('=' * 60)
print('Test: Non-condo property (should PASS regardless)')
print('=' * 60)

pass_result = z3_adapter.validate_rule(
    rule_description=condo_rule_description,
    rule_type='condo_fee_z3',
    extraction_results={},
    document_text=(
        'Property Type: Single Family Home\n'
        'No condo fees applicable'
    ),
)
print(f"Recommendation: {pass_result['recommendation']}")
print(f"Reasoning: {pass_result['reasoning']}")